# Decision Tree
Decision tree is a supervised machine learning algorithm used for classification and regression. A binary tree is constructed, where each internal node represents a feature, and each leaf node represents a prediction or outcome. The tree segment the predictor space (see image below). The tree is built by recursively splitting the dataset based on the best attribute that maximizes information gain or minimizes impurity. When a prediction is needed, the input record traverses the tree from the root to a leaf, following the path determined by the attribute values, and the corresponding prediction is made based on the outcome associated with that leaf node.

The classical decision tree algorithms have been around for decades, and they are the basis of more powerful techniques such as random forest.


![image.png](https://raw.githubusercontent.com/IIB-Lab/AI-ML-bootcamp-tutorials/refs/heads/main/02-supervised-learning/knn-and-decision-trees/img/5.webp)



### Import libraries

In [ ]:
import numpy as np
import pandas as pd

### Read data
We will use the digit dataset in this tutorial. Each value in the `X` (features) represent the pixel density.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 1. Load the dataset
digits = load_digits()

# 2. Extract features (X) and target labels (y)
# X contains the flattened 8x8 images (64 features per sample)
# y contains the actual true digit (0 through 9)
X = digits.data
y = digits.target

# 3. Split into training and testing sets 
# Using a fixed random_state ensures your comparisons are fair and reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

# Optional: Print the shapes to verify
print(f"Total dataset shape: {X.shape}")
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

plt.figure(1, figsize=(3, 3))
plt.imshow(digits.images[10], cmap=plt.cm.gray_r, interpolation="nearest")

plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(1))
plt.gca().yaxis.set_major_locator(ticker.MultipleLocator(1))

# By default, ticks center on pixels, so we shift them to the edges (-0.5)
plt.gca().xaxis.set_ticks(np.arange(-0.5, 8, 1), minor=True)
plt.gca().yaxis.set_ticks(np.arange(-0.5, 8, 1), minor=True)
plt.gca().grid(which='minor', color='k', linestyle='-', linewidth=1.5, alpha=0.1)


# plt.scatter(4, 4, color='red', marker=',', s=60, facecolors='none', edgecolors='red', linewidths=2)

plt.show()



## Decision tree concept

The process of building a decision tree mainly involves two steps:
1. Dividing the predictor space into several distinct, non-overlapping regions
2. Using the most-common class label for the region of new observation as the prediction of the new observation

![image.png](https://raw.githubusercontent.com/IIB-Lab/AI-ML-bootcamp-tutorials/refs/heads/main/02-supervised-learning/knn-and-decision-trees/img/6.webp)

In order to split the predictor space into distinct regions, we use binary recursive splitting, which grows our decision tree until we reach a stopping criterion.

![image.png](https://raw.githubusercontent.com/IIB-Lab/AI-ML-bootcamp-tutorials/refs/heads/main/02-supervised-learning/knn-and-decision-trees/img/5.webp)

In a decision tree, every node will have two children. The children can be leaf nodes or sub-trees, which enhances the prediction of the previous node. This goes until we get to a node that does not split. This last node is known as a leaf node.

For example:

![image.png](https://raw.githubusercontent.com/IIB-Lab/AI-ML-bootcamp-tutorials/refs/heads/main/02-supervised-learning/knn-and-decision-trees/img/7.png)

Our algorithm's goal is to find the best variables (features) and cutoff values for each decision node

### Cost function
"Impurity" means the likelyhood of an incorrect classification when a division is performed. The purpose of cost functions in this algorithm is to evaluate impurity.

Considering the example in our digit dataset.

In [ ]:
prediction = X[:, 36]  == 0
true_label = (y == 0)


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# 2. Compute the confusion matrix array
labels_order = [True, False] 

# 1. Compute the matrix with explicit labels
cm = confusion_matrix(true_label, prediction, labels=labels_order)

# 2. Visualize
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Positive", "Negative"])
disp.plot(cmap=plt.cm.Blues)

### Gini index
The Gini index (also known as Gini impurity) is a popular cost function in decision trees. It is a measure that quantifies the impurity or disorder of a dataset with respect to the class labels.
 The Gini index is calculated as follows:

$$
\mathrm{Gini} = 1 - \sum_{i=1}^{n} (p_i)^2
 $$
where $p_i$ is the probability of having that class $i$. 


For example, if we have 8 cats and 2 dogs,  $p_\mathrm{cat} = 0.8$, $p_\mathrm{dog} = 0.2$, the Gini impurity would be $0.32$ (try calculate this yourself!)

The theoretical maximum is calculated as:
$$\mathrm{Maximum\ Gini} = 1 - \frac{1}{n}$$

* Gini = 0: Perfect purity. Every single data point in that group belongs to the exact same class.

* Higher Gini: More impurity. The data points are a messy mix of different classes.

In [ ]:
def gini_index(y_list, print_demo=False):
  # find the list of categories in the list (which is the list of unique elements in the list)
  category_list = np.unique(y_list)

  # number of total elements in the list
  N = len(y_list)

  gini = 1
  for category in category_list: # go through all categories
    count = np.sum(y_list==category)
    P_i = count / N
    if print_demo:
      print(f"Category: {category}, Count: {count}, p_i: {P_i}")

    gini = gini - P_i**2
  return gini

# we can see that the result is very close to 0.5 because the list is very impure
print("Gini index is: ", gini_index(y,   print_demo=True) )

### Entropy
Another way to measure impurity is entropy. In information theory, entropy describes the average level of information or uncertainty and can be defined as the following.
$$ S = -\sum_{i=1}^{n} p_i \log_2 p_i$$


For example, if we have 8 cats and 2 dogs,  $p_\mathrm{cat} = 0.8$, $p_\mathrm{dog} = 0.2$, the entropy would be $0.72$ (try calculate this yourself!)

The theoretical maximum entropy is calculated as $\log_2(n)$. For 10 classes (like our digits dataset): $\log_2(10) \approx 3.32$

Later, when making a decision tree split, entropy is used to evaluate the potential splits based on different features. The split that results in the lowest entropy (i.e., the greatest reduction in impurity) is chosen.

In [ ]:
def entropy(y_list, print_demo=False):
    # find the list of categories in the list (which is the list of unique elements in the list)
    category_list = np.unique(y_list)

    # number of total elements in the list
    N = len(y_list)

    entropy_val = 0
    for category in category_list: # go through all categories
        count = np.sum(y_list == category)
        P_i = count / N

        if print_demo:
            print(f"Category: {category}, Count: {count}, P_i: {P_i}")

        entropy_val = entropy_val - P_i * np.log2(P_i)

    return entropy_val

print( entropy(y)  )

In [ ]:
def create_split(array, threshold):

    # Find all indices of elements in array that are below the threshold
    left_indices = np.argwhere(array <= threshold).flatten()

    # Find all indices of elements in array that are above the threshold
    right_indices = np.argwhere(array > threshold).flatten()

    return left_indices, right_indices

example_list = np.array([14, 58, 84, 23, 99, 73])
create_split(example_list, threshold=50)

### Information Gain

As we have seen, cuts are compared by impurity. Therefore, we are interested in comparing those cuts that generate less impurity. We use information gain for this. It indicates the improvement when making different partitions and is usually used with entropy. If we were to apply the exact same logic to the Gini index, the resulting metric is instead referred to as Gini Gain or the reduction in Gini impurity.

Information gain is calculated by measuring the reduction in entropy (or increase in purity) achieved after splitting the data based on a specific feature. It is calculated as the difference between the entropy of the parent node before the split and the weighted average of the entropies of the child nodes after the split.

$$ \text{Information Gain}(D, A) = \text{Entropy}(D) - \sum_{v \in \text{Values}(A)} \frac{|D_v|}{|D|} \times \text{Entropy}(D_v)
$$

$$
\begin{align*}
D &: \text{dataset at the current node.} \\
A &: \text{feature being considered for splitting.} \\
\text{Values}(A) &: \text{possible values that feature } A \text{ can take.} \\
D_v &: \text{subset of data where feature } A \text{ takes the value } v.
\end{align*}
$$

The higher the Information Gain for a feature, the more it helps in reducing uncertainty in the target variable. Later, our algorithm will select the feature that offers the highest Information Gain as the splitting criterion at each node.


In [ ]:
def information_gain(X_one_feature_only, y, threshold):
    # calculate the entropy before the split
    parent_entropy = entropy(y)

    # find the indices of elements above and below the threshold so we can make the split
    left_indices, right_indices = create_split(X_one_feature_only, threshold)

    left_elements = y[left_indices]
    right_elements = y[right_indices]

    # find how many elements are there before and after the split
    n = len(y)
    n_left = len(left_indices)
    n_right = len(right_indices)

    # if the split resulted in 0 item in either side (meaning all elements went into one side and the other side is empty)
    # then the information gain is zero because the split is useless (not a split at all)
    if n_left == 0 or n_right == 0:
        return 0

    # calculate the entropy of two sides
    left_child_entropy = entropy(left_elements)
    right_child_entropy = entropy(right_elements)

    # take the weighted average
    child_entropy = (n_left/n)*left_child_entropy + (n_right/n)*right_child_entropy

    return parent_entropy - child_entropy

# example of a perfect cut:
X_example = np.array([1,2,3,4,5,  6,7,8,9,10]) # example feature
y_example = np.array([0,0,0,0,0,  1,1,1,1,1]) # two classes
information_gain(X_example, y_example, 5)

An information gain of 1 would be the best possible result for this simple example that only has two classes, 0 and 1, meaning there is a perfect separation of the target variable into its classes. This means that after the split, each subset contains only instances of a single class, completely reducing the uncertainty. In practical scenarios, achieving an information gain of 1 is uncommon, because real-world data that have noise, overlapping classes, or other complexities.

Now, going back to our digit example, let's calcuate the information gain based on the 37th pixel density and calcuate information gain!

In [ ]:
p37x = X[:, 36]
y0 = y == 0

print(information_gain(p37x, y0, 0))

### Finding the best variable and threshold for making splits

To make a split, we have 2 steps:
1. Calculate the information gain for possible threshold values of all features (variables).
2. Choose the split that generates the highest information gain as a split.

It uses brute-force to find the best split!


In [ ]:
def max_information_gain_split(X, y,):
    # initialize the max info gain, feature, and threshold as void values
    best_info_gain = -1
    best_feature =  None
    best_threshold = None

    # go over all features
    for feature in range(X.shape[1]):
        # X_of_feature will be an array of all values of the selected feature
        X_of_feature = X[:, feature]


        # we use `np.unique` to find all possible values which we can use as the threshold, and go over each one
        possible_threshold_options = np.unique(X_of_feature)
        for threshold in possible_threshold_options:
            # calculate the information gain of this specific threshold for this specific feature
            info_gain = information_gain(X_of_feature, y, threshold)

            # keep this as the `best` one if it have info_gain higher than the existing best_info_gain
            if  info_gain > best_info_gain:
                best_info_gain = info_gain
                best_feature =  feature
                best_threshold = threshold

    return best_feature, best_threshold, best_info_gain


best_feature, best_threshold, best_info_gain  = max_information_gain_split(X,y)


print(f"The best split for pixel idx {best_feature} = {best_threshold}. The info gain is {best_info_gain}")

As we can see, the variable with the highest Information Gain is Weight. Therefore, it will be the variable that we first use to do the split. In addition, we also have the value on which the split must be performed: 102.

To demostrate `max_information_gain_split` in a more simplified way, we can use the following figure to show the information gain for different split using a set of simplified data.


![image.png](https://raw.githubusercontent.com/IIB-Lab/AI-ML-bootcamp-tutorials/refs/heads/main/02-supervised-learning/knn-and-decision-trees/img/8.png)


### Decision tree algorithm
The main algorithm can be divided into three steps:
1. Initialization of parameters (e.g. maximum depth, minimum samples per split)
2. Building the decision tree, involving binary recursive splitting, evaluating each possible split at the current stage, and continuing to grow the tree until a stopping criterion is satisfied
3. Making a prediction, which can be described as traversing the tree recursively and returning the most-common class label as a response value.

In our tree, there are two kinds of nodes: leaf nodes and subtrees. We will store them as python dictionaries.

For example:

```python
example_leaf_node = {
    "is_leaf":True,
    "value": 0 # the most_common_Label here
}

example_subtree = {
    "is_leaf":False,
    "feature": 'Weight',
    "threshold": 103,
    "left": example_leaf_node, # can be any leaf node or subtree
    "right": {} # can be any leaf node or subtree
}
```

### Building a tree and determining the depth of the tree
We use binary recursive splitting to build a tree. We first evaluate every possible split for every feature to select the best place to split, and make the split. Then, we use recursion to repeat, continuing to grow the tree until we meet a stopping criterion.

Without a stopping mechanism in place, we would have created an endless loop which the program keeps on splitting forever. We will use three ways to limit the tree depth:

- `max_depth`: maximum depth of the tree.
- `min_samples_split`: indicates the minimum number of observations a sheet must have to continue creating new nodes.
- `min_information_gain`: the minimum amount the information gain must increase for the tree to continue growing.

Once we satisfy the stopping criteria the method will recursively return all nodes, allowing us to build a full-grown decision tree.


In [ ]:
def build_tree(X, y, max_depth=100, min_samples_split=2, min_info_gain=0.00001, recursion_depth_counter=0):
    n_samples, n_features = X.shape
    n_class_labels = len(np.unique(y))

    # stopping criteria: max depth, min sample split conditions, or pure node
    if ((recursion_depth_counter >= max_depth) # Usually >= is safer for depth
        or (n_samples < min_samples_split) 
        or (n_class_labels <= 1)):

        most_common_Label = np.argmax(np.bincount(y))
        return {"is_leaf": True, "value": most_common_Label}

    # get best split
    best_feat, best_thresh, best_info_gain = max_information_gain_split(X, y)

    # stopping criteria: the best info gain is too small
    if best_info_gain < min_info_gain:
        most_common_Label = np.argmax(np.bincount(y))
        return {"is_leaf": True, "value": most_common_Label}

    # make split
    left_idx, right_idx = create_split(X[:, best_feat], best_thresh)

    # # Check if the split resulted in empty arrays
    # if len(left_idx) == 0 or len(right_idx) == 0:
    #     most_common_Label = np.argmax(np.bincount(y))
    #     return {"is_leaf": True, "value": most_common_Label}


    # grow children recursively
    left_child = build_tree(X[left_idx], y[left_idx], max_depth, min_samples_split, min_info_gain, recursion_depth_counter + 1)
    right_child = build_tree(X[right_idx], y[right_idx], max_depth, min_samples_split, min_info_gain, recursion_depth_counter + 1)
    
    return {
        "is_leaf": False, 
        "feature": best_feat, 
        "threshold": best_thresh, 
        "left": left_child, 
        "right": right_child 
    }



max_depth = 5
min_samples_split = 20
min_information_gain = 0.0001

trained_tree = build_tree(X_train,y_train, max_depth, min_samples_split,min_information_gain)
display(trained_tree)

![image.png](https://raw.githubusercontent.com/IIB-Lab/AI-ML-bootcamp-tutorials/refs/heads/main/02-supervised-learning/knn-and-decision-trees/img/9.png)


### Making Prediction using the tree
Making a prediction can be implemented by recursively traversing the tree. For every sample in our dataset, we compare the node feature and threshold values to the current sample's values and decide if we have to take a left or a right turn.

Once we reach a leaf node we simply return the most common class label as our prediction.

In [ ]:
def traverse_tree(x, node):
    if node["is_leaf"]:
        return node["value"]

    feature = node["feature"]
    threshold =  node["threshold"]
    if x[feature] <= threshold:
        return traverse_tree(x, node["left"])
    else:
        return traverse_tree(x, node["right"])

def predict(x, decision_tree):
    return traverse_tree(x, decision_tree)

# predict the class for person #20
print(predict(X_test[20], trained_tree))

# the actual class for person #20
print(y_test[20])

### Evaluate accuracy

Finally, we run the `predict` function on all of the samples in the test dataset, and compare it againsst the actual class label.

In [ ]:
correct_predictions = 0
total_predictions = len(y_test)

for i in range(len(y_test)):
    predicted_class = predict(X_test[i], trained_tree)
    actual_class = y_test[i]

    if predicted_class == actual_class:
        correct_predictions += 1

accuracy = correct_predictions / total_predictions
print(f"Accuracy: {accuracy}")

**Use `sklearn`'s implementation**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Initialize the Decision Tree Classifier
sk_tree = DecisionTreeClassifier(
    criterion="entropy",           # Uses Information Gain/Entropy to measure split quality
    max_depth=5,                   # Restricts tree depth to prevent overfitting
    min_samples_split=20,          # Minimum samples required to split an internal node
    min_impurity_decrease=0.0001     # Minimum impurity drop required for a split (min InfoGain)
)
sk_tree.fit(X_train, y_train)
sk_accruacy = accuracy_score(y_test, sk_tree.predict(X_test))
print(f"Accuracy using sklearn: {sk_accruacy}")



**Comparing accuracy**

Now let's compare the model accuracy between the two models using visualization.

In [ ]:

categories = ['Manual implementation', 'sklearn\'s implementation',]
values = [accuracy, sk_accruacy]

plt.figure(figsize=(7, 5))
bars = plt.bar(categories, values, color=['skyblue', 'lightgreen', ])

# Add values on top of the bars
plt.bar_label(bars, padding=3)

plt.title('Comparison accuracy of the Decision Tree classifier')
plt.ylabel('Values')

plt.ylim(0, max(values) * 1.2) # Add some space above the bars for the labels



### Exercise:
0. Re-run the code with `max_depth` set to 2. Then, re-run the code with `max_depth` set to 20. Write down how it affected the accuracy and the tree.
1. Modify code so it uses the Gini index in cost function instead of the entropy function. Write down how it affected the accuracy.
2. Is it possible for information gain of a split to be 0? if so, what does it mean and how should our program handle this case?


## References
- https://towardsdatascience.com/implementing-a-decision-tree-from-scratch-f5358ff9c4bb
- https://anderfernandez.com/en/blog/code-decision-tree-python-from-scratch/
- https://towardsai.net/p/programming/decision-trees-explained-with-a-practical-example-fe47872d3b53
- https://www.datacamp.com/tutorial/decision-tree-classification-python
- https://towardsdatascience.com/decision-tree-algorithm-in-python-from-scratch-8c43f0e40173